In [1]:
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

In [2]:
class VGG16(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        num_features = self.model.classifier[6].in_features
        self.model.classifier[6] = nn.Linear(num_features, 2) 
        
    def forward(self, x):
        return self.model(x)

In [3]:
class VGG11(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.vgg11(weights=models.VGG11_Weights.IMAGENET1K_V1)
        num_features = self.model.classifier[6].in_features
        self.model.classifier[6] = nn.Linear(num_features, 2)  

    def forward(self, x):
        return self.model(x)

In [4]:
class DistillLoss(nn.Module):
    def __init__(self, T=4.0, alpha=0.7):
        super().__init__()
        self.T = T
        self.alpha = alpha
        self.kld = nn.KLDivLoss(reduction='batchmean')
        self.ce = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, true_labels):
        kd = self.kld(F.log_softmax(student_logits / self.T, dim=1),
                      F.softmax(teacher_logits / self.T, dim=1)) * (self.T ** 2)
        ce = self.ce(student_logits, true_labels)
        return self.alpha * kd + (1 - self.alpha) * ce

In [5]:
from torchvision import datasets

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("/kaggle/input/fakeface-train-data-v2", transform=transform)
val_dataset   = datasets.ImageFolder("/kaggle/input/fakeface-valid-data-v2", transform=transform)
test_dataset = datasets.ImageFolder("/kaggle/input/fakeface-test-data-v2", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)
test_loader   = DataLoader(test_dataset, batch_size=64, shuffle=False,num_workers=2)

In [6]:
import torch
from collections import OrderedDict

def load_teacher_model(path, device="cuda"):
    model = VGG16()  
    state_dict = torch.load(path, map_location=device)

    new_state_dict = OrderedDict((k.replace("module.", ""), v) for k, v in state_dict.items())
    model.load_state_dict(new_state_dict)

    model.to(device)
    model.eval()  
    return model

In [7]:
teacher = load_teacher_model("/kaggle/input/teacher-models/vgg16.pth", device="cuda")
student = VGG11()

if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPU với DataParallel.")
    student = nn.DataParallel(student)
    teacher = nn.DataParallel(teacher)
    
kd_loss_fn = DistillLoss(T=4.0, alpha=0.7)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 229MB/s]
Downloading: "https://download.pytorch.org/models/vgg11-8a719046.pth" to /root/.cache/torch/hub/checkpoints/vgg11-8a719046.pth
100%|██████████| 507M/507M [00:02<00:00, 229MB/s]


Sử dụng 2 GPU với DataParallel.


In [8]:
student = student.to(device)
teacher = teacher.to(device)

In [9]:
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

def evaluate_model_on_validation(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = float('nan')

    return acc, auc  

In [10]:
import time
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from tqdm import tqdm  

def train_student_kd_with_validation(student, teacher, train_loader, val_loader, kd_loss_fn, optimizer, device, epochs=10):
    ce_loss_fn = nn.CrossEntropyLoss()
    start_training = time.time()  

    for epoch in range(epochs):
        student.train()
        teacher.eval()

        total_kd_loss = 0
        total_student_loss = 0
        total_teacher_loss = 0

        all_probs = []
        all_labels = []

        start_epoch = time.time()

        progress_bar = tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training", leave=False)

        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)
        
            optimizer.zero_grad()  
        
            with torch.no_grad():
                t_logits = teacher(x)
                teacher_ce_loss = ce_loss_fn(t_logits, y)  
        
            s_logits = student(x)
        
            student_ce_loss = ce_loss_fn(s_logits, y)    
            kd_loss = kd_loss_fn(s_logits, t_logits, y)   
        
            kd_loss.backward()
            optimizer.step()
        
            total_kd_loss += kd_loss.item()
            total_student_loss += student_ce_loss.item()
            total_teacher_loss += teacher_ce_loss.item()

            probs = torch.softmax(s_logits, dim=1)[:, 1].detach().cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y.cpu().numpy())

            progress_bar.set_postfix({
                "KD": f"{kd_loss.item():.4f}",
                "StudentCE": f"{student_ce_loss.item():.4f}"
            })

        try:
            train_auc = roc_auc_score(all_labels, all_probs)
        except:
            train_auc = float('nan')

        epoch_time = time.time() - start_epoch

        val_acc, val_auc = evaluate_model_on_validation(student, val_loader, device)

        print(f"[Epoch {epoch+1}] "
              f"KD Loss: {total_kd_loss / len(train_loader):.4f} | "
              f"Train AUC: {train_auc:.4f} | "
              f"Val AUC: {val_auc:.4f} | Val ACC: {val_acc:.4f} | "
              f"Time: {epoch_time:.2f}s")

    total_time = time.time() - start_training
    print(f"⏱️ Tổng thời gian training: {total_time:.2f} giây ({total_time/60:.2f} phút)")


In [11]:
from sklearn.metrics import classification_report
import torch

def evaluate_models_report(student, teacher, dataloader, device):
    student.eval()
    teacher.eval()

    all_labels = []
    student_preds = []
    teacher_preds = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            s_logits = student(x)
            t_logits = teacher(x)

            student_cls = torch.argmax(s_logits, dim=1).cpu().numpy()
            teacher_cls = torch.argmax(t_logits, dim=1).cpu().numpy()
            true_labels = y.cpu().numpy()

            student_preds.extend(student_cls)
            teacher_preds.extend(teacher_cls)
            all_labels.extend(true_labels)

    print("📘 [Student Model] Classification Report:")
    print(classification_report(all_labels, student_preds, digits=4))

    print("📗 [Teacher Model] Classification Report:")
    print(classification_report(all_labels, teacher_preds, digits=4))


In [12]:
train_student_kd_with_validation(student, teacher, train_loader, val_loader, kd_loss_fn, optimizer, device, epochs=10)

[Epoch 1] KD Loss: 1.1032 | Train AUC: 0.9829 | Val AUC: 0.9926 | Val ACC: 0.9587 | Time: 1078.87s


[Epoch 2] KD Loss: 0.1860 | Train AUC: 0.9999 | Val AUC: 0.9943 | Val ACC: 0.9661 | Time: 1093.15s


[Epoch 3] KD Loss: 0.1219 | Train AUC: 1.0000 | Val AUC: 0.9946 | Val ACC: 0.9674 | Time: 1093.88s


[Epoch 4] KD Loss: 0.1033 | Train AUC: 0.9999 | Val AUC: 0.9940 | Val ACC: 0.9677 | Time: 1093.49s


[Epoch 5] KD Loss: 0.1228 | Train AUC: 0.9999 | Val AUC: 0.9956 | Val ACC: 0.9720 | Time: 1092.73s


[Epoch 6] KD Loss: 0.0756 | Train AUC: 1.0000 | Val AUC: 0.9936 | Val ACC: 0.9675 | Time: 1092.30s


[Epoch 7] KD Loss: 0.1248 | Train AUC: 0.9998 | Val AUC: 0.9966 | Val ACC: 0.9780 | Time: 1091.32s


[Epoch 8] KD Loss: 0.0511 | Train AUC: 1.0000 | Val AUC: 0.9963 | Val ACC: 0.9767 | Time: 1091.74s


[Epoch 9] KD Loss: 0.0459 | Train AUC: 1.0000 | Val AUC: 0.9963 | Val ACC: 0.9763 | Time: 1091.83s


[Epoch 10] KD Loss: 0.1166 | Train AUC: 0.9997 | Val AUC: 0.9965 | Val ACC: 0.9769 | Time: 1090.52s
⏱️ Tổng thời gian training: 11954.19 giây (199.24 phút)


In [13]:
evaluate_models_report(student, teacher, test_loader, device)

📘 [Student Model] Classification Report:
              precision    recall  f1-score   support

           0     0.9715    0.9805    0.9760     20000
           1     0.9803    0.9712    0.9757     20000

    accuracy                         0.9758     40000
   macro avg     0.9759    0.9758    0.9758     40000
weighted avg     0.9759    0.9758    0.9758     40000

📗 [Teacher Model] Classification Report:
              precision    recall  f1-score   support

           0     0.9619    0.9831    0.9724     20000
           1     0.9827    0.9610    0.9718     20000

    accuracy                         0.9721     40000
   macro avg     0.9723    0.9721    0.9721     40000
weighted avg     0.9723    0.9721    0.9721     40000



In [14]:
torch.save(student.state_dict(), "/kaggle/working/vgg11_student.pth")